# Notebook 04 — Escalamiento y PCA
## Proyecto Integrador | Minería de Datos I
**Integrante:** Val Martinetti

---
### Objetivo
Reducir la dimensionalidad del dataset usando **Análisis de Componentes Principales (PCA)**, previo escalamiento obligatorio. Se interpretarán los componentes resultantes en términos de comportamiento de usuario.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
import warnings
warnings.filterwarnings('ignore')

sns.set_theme(style='whitegrid')

# Cargar dataset limpio
df = pd.read_csv('../data/processed/streaming_users_clean.csv',
                 parse_dates=['last_login_date'])
print(f"Dataset: {df.shape[0]} filas × {df.shape[1]} columnas")
df.head()

## 1. Selección de variables para PCA

In [ ]:
# PCA requiere variables numéricas continuas
# Variables disponibles: age, monthly_watch_time_mins, customer_support_tickets
features = ['age', 'monthly_watch_time_mins', 'customer_support_tickets']
X = df[features].copy()

print("Variables seleccionadas para PCA:")
for col in features:
    print(f"  {col}: {X[col].dtype} — media={X[col].mean():.2f}, std={X[col].std():.2f}")

print()
print("Justificación de la selección:")
print("  - age:  describe el perfil demográfico del usuario")
print("  - monthly_watch_time_mins: describe el comportamiento de consumo")
print("  - customer_support_tickets: describe la interacción con el servicio de soporte")
print()
print("Variables excluidas:")
print("  - user_id: identificador, no tiene valor analítico")
print("  - subscription_plan/country/genre: categóricas — requieren codificación numérica")
print("  - last_login_date: con 769 NaT; no se usa en este análisis numérico")

## 2. Diagnóstico previo: ¿por qué escalar?

In [ ]:
# Comparar varianzas y escalas
print("Varianza por variable ANTES de escalar:")
print(X.var().round(2))
print()
print("Rango por variable:")
for col in features:
    print(f"  {col}: [{X[col].min():.1f}, {X[col].max():.1f}]")
print()
print("Sin escalar, monthly_watch_time_mins dominaría el PCA")
print("porque su varianza es ~10.000x mayor que la de tickets.")

**Por qué es obligatorio escalar antes de PCA:**  
PCA maximiza la varianza explicada. Sin escalar, `monthly_watch_time_mins` (varianza en el orden de decenas de miles) domina completamente las componentes, haciendo que las otras dos variables sean ignoradas. Escalar lleva todas las variables a media=0 y varianza=1, de modo que **cada variable contribuya de forma equitativa** basada en su contenido informativo, no en sus unidades de medida.

## 3. Escalamiento con StandardScaler (Z-score)

In [ ]:
scaler = StandardScaler()
X_scaled = pd.DataFrame(
    scaler.fit_transform(X),
    columns=features,
    index=X.index
)

print("Varianza por variable DESPUÉS de estandarizar (todas = 1.0):")
print(X_scaled.var().round(4))
print()
print("Media (todas ≈ 0):")
print(X_scaled.mean().round(6))

## 4. Aplicación de PCA

In [ ]:
# PCA completo (tantos componentes como variables: 3)
pca_full = PCA(n_components=3, random_state=42)
pca_full.fit(X_scaled)

# Varianza explicada
varianza_expl = pca_full.explained_variance_ratio_
varianza_acum = np.cumsum(varianza_expl)

print("Varianza explicada por componente:")
for i, (var, acum) in enumerate(zip(varianza_expl, varianza_acum), 1):
    print(f"  PC{i}: {var*100:.2f}% (acumulada: {acum*100:.2f}%)")

In [ ]:
# Gráfico de varianza explicada (Scree Plot)
fig, axes = plt.subplots(1, 2, figsize=(11, 4))

# Scree plot
axes[0].bar(range(1, 4), varianza_expl * 100, color=['#4e79a7','#f28e2b','#59a14f'], alpha=0.85)
axes[0].plot(range(1, 4), varianza_expl * 100, 'ko-', markersize=6)
axes[0].set_xticks(range(1, 4))
axes[0].set_xticklabels(['PC1', 'PC2', 'PC3'])
axes[0].set_ylabel('Varianza explicada (%)')
axes[0].set_title('Scree Plot')

# Varianza acumulada
axes[1].plot(range(1, 4), varianza_acum * 100, 'bo-', markersize=8)
axes[1].axhline(y=80, color='red', linestyle='--', alpha=0.7, label='80% umbral')
axes[1].set_xticks(range(1, 4))
axes[1].set_xticklabels(['PC1', 'PC2', 'PC3'])
axes[1].set_ylabel('Varianza acumulada (%)')
axes[1].set_title('Varianza acumulada')
axes[1].legend()

plt.tight_layout()
plt.savefig('../reports/fig09_pca_scree.png', dpi=100, bbox_inches='tight')
plt.show()

**Interpretación del Scree Plot:**  
- **PC1** explica ~34% de la varianza total.
- **PC2** explica ~33% adicional.
- **PC3** explica el ~33% restante.  

Las tres componentes tienen contribuciones casi iguales (~33% cada una). Esto se debe a que las variables de entrada están prácticamente **no correlacionadas entre sí** (correlaciones ≈ 0, verificadas en el EDA). Cuando las variables son independientes, PCA no puede reducir dimensionalidad de forma eficiente porque cada variable ya es, por sí sola, una componente principal. Se necesitarán las **2 primeras PCs** para superar el 67% de varianza explicada.

## 5. Elección del número de componentes

In [ ]:
# Con 2 componentes superamos el 67% de varianza
# No hay un umbral mágico, pero 80% es frecuente en la práctica
pca = PCA(n_components=2, random_state=42)
X_pca = pca.fit_transform(X_scaled)
X_pca_df = pd.DataFrame(X_pca, columns=['PC1', 'PC2'], index=df.index)

print(f"Varianza explicada con 2 componentes: {pca.explained_variance_ratio_.sum()*100:.1f}%")
print()
print("Decisión: se retienen 2 componentes.")
print("Justificación:")
print("  - Con 3 componentes se retiene el 100% (sin reducción real).")
print("  - La baja correlación entre variables limita la compresión posible.")
print("  - 2 componentes permiten visualización 2D y retienen 67% de información.")

## 6. Interpretación de las cargas (loadings)

In [ ]:
# Matriz de cargas (loadings): contribución de cada variable a cada componente
loadings = pd.DataFrame(
    pca.components_,
    columns=features,
    index=['PC1', 'PC2']
)
print("Matriz de cargas (loadings):")
print(loadings.round(4))
print()
print("Interpretación:")
for pc in ['PC1', 'PC2']:
    principal = loadings.loc[pc].abs().idxmax()
    valor = loadings.loc[pc, principal]
    print(f"  {pc}: carga más alta en '{principal}' ({valor:.3f})")

In [ ]:
# Visualización de loadings
fig, ax = plt.subplots(figsize=(7, 5))
loadings.T.plot(kind='bar', ax=ax, colormap='Set2', alpha=0.85, edgecolor='white')
ax.axhline(y=0, color='black', linewidth=0.8)
ax.set_ylabel('Carga (loading)')
ax.set_title('Cargas de las variables en PC1 y PC2')
ax.tick_params(axis='x', rotation=20)
ax.legend(title='Componente')
plt.tight_layout()
plt.savefig('../reports/fig10_pca_loadings.png', dpi=100, bbox_inches='tight')
plt.show()

**Interpretación de los componentes:**
- **PC1 — "Dimensión de consumo":** cargada principalmente por `monthly_watch_time_mins`. Representa qué tan intensivo es el usuario en su consumo de contenido. Puntuación alta en PC1 → heavy user.
- **PC2 — "Dimensión demográfico-soporte":** combina `age` y `customer_support_tickets`. Representa el perfil demográfico con demanda de soporte técnico. Puntuación alta en PC2 → usuario mayor con más necesidades de soporte.

## 7. Visualización en espacio PCA

In [ ]:
# Unir scores PCA con atributos del dataset
df_pca = X_pca_df.join(df[['subscription_plan', 'age', 'monthly_watch_time_mins']])

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Scatter coloreado por plan
palette = {'Básico': '#4e79a7', 'Estándar': '#f28e2b', 'Premium': '#59a14f'}
for plan, grupo in df_pca.groupby('subscription_plan'):
    axes[0].scatter(grupo['PC1'], grupo['PC2'],
                    label=plan, color=palette[plan], alpha=0.3, s=10)
axes[0].set_xlabel(f"PC1 ({pca.explained_variance_ratio_[0]*100:.1f}% varianza)")
axes[0].set_ylabel(f"PC2 ({pca.explained_variance_ratio_[1]*100:.1f}% varianza)")
axes[0].set_title('Espacio PCA coloreado por plan')
axes[0].legend(title='Plan')

# Scatter coloreado por edad (continua)
sc = axes[1].scatter(df_pca['PC1'], df_pca['PC2'],
                     c=df_pca['age'], cmap='viridis', alpha=0.3, s=10)
plt.colorbar(sc, ax=axes[1], label='Edad (años)')
axes[1].set_xlabel(f"PC1 ({pca.explained_variance_ratio_[0]*100:.1f}% varianza)")
axes[1].set_ylabel(f"PC2 ({pca.explained_variance_ratio_[1]*100:.1f}% varianza)")
axes[1].set_title('Espacio PCA coloreado por edad')

plt.tight_layout()
plt.savefig('../reports/fig11_pca_scatter.png', dpi=100, bbox_inches='tight')
plt.show()

**Interpretación del espacio PCA:**
- **Por plan:** los usuarios Premium se desplazan hacia la derecha en PC1 (mayor consumo), confirmando el hallazgo del EDA. Los planes Básico y Estándar se superponen más, distinguiéndose principalmente en PC1.
- **Por edad:** el gradiente vertical en PC2 muestra que la edad se distribuye a lo largo de este eje, con usuarios de mayor edad más arriba (PC2 positivo). La separación no es perfecta, coherente con la baja correlación hallada en el EDA.

## Síntesis del análisis PCA

In [ ]:
print("""
RESUMEN DEL ANÁLISIS DE ESCALAMIENTO Y PCA
═══════════════════════════════════════════════════════════

ESCALAMIENTO:
  - Técnica: StandardScaler (Z-score)
  - Motivo: varianzas muy dispares antes de escalar
    (watch_time ~miles > age ~200 > tickets ~1)
  - Post-escalado: todas las variables con varianza = 1

PCA:
  - Variables de entrada: age, monthly_watch_time_mins, tickets
  - Componentes retenidos: 2 (de 3 posibles)
  - Varianza explicada: ~67%
  - PC1 (~34%): dimensión de consumo (watch_time dominante)
  - PC2 (~33%): dimensión demográfico-soporte (age + tickets)

LIMITACIÓN PRINCIPAL:
  Las variables de entrada tienen correlaciones cercanas a 0
  entre sí (verificado en EDA). Esto limita la efectividad
  del PCA: al ser independientes, no hay redundancia que
  comprimir. Para un PCA más informativo, incorporar
  variables derivadas o codificadas (p.ej. plan_encoded)
  podría aportar más estructura al espacio reducido.

VALOR DEL ANÁLISIS:
  El PCA confirma que los 3 ejes de variación del dataset
  son ortogonales (independientes), lo que valida la calidad
  del EDA y sugiere que no hay multicolinealidad entre las
  variables cuantitativas disponibles.
""")